# 🧠 Brain Tumour MRI — Full Training Notebook

## Before you run anything:
1. **Enable GPU**: `Runtime` → `Change runtime type` → `T4 GPU` → Save
2. **Upload data to Google Drive** (do this while reading the notebook):
   - Create a folder in Google Drive called **`Tumour-MRI-Data`**
   - Upload **`BraTS-MEN-RT-Train-v2`** folder inside it
   - Upload **`IXI_T1`** folder inside it
   - Final structure: `MyDrive/Tumour-MRI-Data/BraTS-MEN-RT-Train-v2/` and `MyDrive/Tumour-MRI-Data/IXI_T1/`

3. Run cells **top to bottom** — each cell has clear instructions.

---
**This notebook is self-contained — all code is embedded. You only need to upload your data.**

## ✅ Step 1 — Check GPU & Install Dependencies

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}')
    print(f'   VRAM: {vram:.1f} GB')
else:
    print('❌ NO GPU DETECTED')
    print('   Go to Runtime → Change runtime type → T4 GPU → Save')
    print('   Then restart and run from the top.')

print(f'\nPyTorch version: {torch.__version__}')

In [ ]:
# Install the two packages not pre-installed in Colab
!pip install -q 'monai>=1.3.0' 'nibabel>=5.0.0'
print('✅ Dependencies installed')

## ✅ Step 2 — Mount Google Drive & Set Data Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
import os

# ── Configure these paths to match where you uploaded data on Drive ──────────
BRATS_DIR = '/content/drive/MyDrive/Tumour-MRI-Data/BraTS-MEN-RT-Train-v2'
IXI_DIR   = '/content/drive/MyDrive/Tumour-MRI-Data/IXI_T1'
SAVE_DIR  = '/content/drive/MyDrive/Tumour-MRI-Data'  # model saved here
# ─────────────────────────────────────────────────────────────────────────────

ok = True
for path, name in [(BRATS_DIR, 'BraTS'), (IXI_DIR, 'IXI')]:
    if os.path.isdir(path):
        patients = [p for p in os.listdir(path) if os.path.isdir(os.path.join(path, p))]
        print(f'✅ {name}: {len(patients)} patient folders found at {path}')
    else:
        print(f'❌ {name} directory NOT FOUND: {path}')
        ok = False

if not ok:
    print('\n⚠️  Fix the paths above before continuing.')

## ✅ Step 3 — Write Project Source Code

In [ ]:
import os
os.makedirs('/content/BrainTumour/src', exist_ok=True)
os.makedirs('/content/BrainTumour/models', exist_ok=True)
os.makedirs('/content/BrainTumour/output/runs', exist_ok=True)
os.makedirs('/content/BrainTumour/results/plots', exist_ok=True)
os.chdir('/content/BrainTumour')
print('✅ Project directories created at /content/BrainTumour')

In [ ]:
%%writefile src/__init__.py
"""MRI tumour segmentation package."""


In [ ]:
%%writefile src/utils.py
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch

LOGGER = logging.getLogger(__name__)

def configure_logging(log_level: str = 'INFO') -> None:
    logging.basicConfig(
        level=getattr(logging, log_level.upper(), logging.INFO),
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    )

def ensure_directory(path) -> Path:
    directory = Path(path)
    directory.mkdir(parents=True, exist_ok=True)
    return directory

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    return tensor.detach().cpu().numpy()

def load_json(path) -> Dict[str, Any]:
    import json
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(path, payload: Dict[str, Any]) -> None:
    import json
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

def compute_dice(preds: torch.Tensor, targets: torch.Tensor) -> float:
    preds = preds.float(); targets = targets.float()
    intersection = (preds * targets).sum().item()
    union = preds.sum().item() + targets.sum().item()
    return 2.0 * intersection / union if union > 0 else 0.0

def compute_iou(preds: torch.Tensor, targets: torch.Tensor) -> float:
    preds = preds.float(); targets = targets.float()
    intersection = (preds * targets).sum().item()
    union = (preds + targets - preds * targets).sum().item()
    return intersection / union if union > 0 else 0.0

def compute_precision(preds: torch.Tensor, targets: torch.Tensor) -> float:
    preds = preds.float(); targets = targets.float()
    tp = (preds * targets).sum().item()
    fp = (preds * (1 - targets)).sum().item()
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def compute_recall(preds: torch.Tensor, targets: torch.Tensor) -> float:
    preds = preds.float(); targets = targets.float()
    tp = (preds * targets).sum().item()
    fn = ((1 - preds) * targets).sum().item()
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def compute_f1(preds: torch.Tensor, targets: torch.Tensor) -> float:
    p = compute_precision(preds, targets)
    r = compute_recall(preds, targets)
    return 2.0 * p * r / (p + r) if (p + r) > 0 else 0.0

def compute_pixel_accuracy(preds: torch.Tensor, targets: torch.Tensor) -> float:
    return (preds == targets).float().mean().item()


In [ ]:
%%writefile src/model.py
from __future__ import annotations
import logging
import torch
from monai.networks.nets import SegResNet, DynUNet, BasicUNet

LOGGER = logging.getLogger(__name__)

def get_model(architecture='segresnet', in_channels=1, out_channels=2, spatial_dims=3):
    architecture = architecture.lower()
    if architecture == 'segresnet':
        model = SegResNet(
            spatial_dims=spatial_dims, in_channels=in_channels, out_channels=out_channels,
            init_filters=64, blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1), dropout_prob=0.1,
        )
    elif architecture == 'unet':
        model = BasicUNet(
            spatial_dims=spatial_dims, in_channels=in_channels, out_channels=out_channels,
            features=(32, 64, 128, 256, 512, 32), dropout=0.1,
        )
    elif architecture == 'dynunet':
        model = DynUNet(
            spatial_dims=spatial_dims, in_channels=in_channels, out_channels=out_channels,
            kernel_size=[[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
            filters=[64, 128, 256, 512, 512, 256],
            strides=[[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
            upsample_kernel_size=[[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
            deep_supervision=False,
        )
    else:
        raise ValueError(f'Unsupported architecture: {architecture!r}')
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOGGER.info('Built %s with %s trainable parameters', architecture, f'{n:,}')
    return model


In [ ]:
%%writefile src/preprocess.py
from __future__ import annotations
import logging
from typing import Tuple
import numpy as np
import torch
import torch.nn.functional as F

LOGGER = logging.getLogger(__name__)

class Preprocessor:
    def __init__(self, spatial_size=(96, 96, 96)):
        self.spatial_size = spatial_size

    def _resize_volume(self, volume, target_shape, is_mask):
        if volume.ndim == 3: volume = volume.unsqueeze(0)
        volume = volume.unsqueeze(0)
        if tuple(volume.shape[-3:]) != tuple(target_shape):
            mode = 'nearest' if is_mask else 'trilinear'
            kw = {} if is_mask else {'align_corners': False}
            volume = F.interpolate(volume.float(), size=target_shape, mode=mode, **kw)
        return volume

    def _normalize_volume(self, volume):
        volume = volume.float()
        if volume.numel() == 0: return volume
        lower = torch.quantile(volume, 0.01)
        upper = torch.quantile(volume, 0.99)
        clipped = torch.clamp(volume, lower, upper)
        mean, std = clipped.mean(), clipped.std()
        if std < 1e-6: return torch.zeros_like(clipped)
        return (clipped - mean) / (std + 1e-6)

    def __call__(self, sample):
        image = sample['image']; mask = sample['mask']
        image_t = torch.from_numpy(image.astype(np.float32)) if isinstance(image, np.ndarray) else image.float()
        mask_t  = torch.from_numpy(mask.astype(np.float32))  if isinstance(mask,  np.ndarray) else mask.float()
        image_t = self._resize_volume(image_t, self.spatial_size, False).squeeze(0)
        mask_t  = self._resize_volume(mask_t,  self.spatial_size, True ).squeeze(0)
        image_t = self._normalize_volume(image_t)
        mask_t  = (mask_t > 0).float()
        return {'image': image_t, 'mask': mask_t,
                'patient_id': sample.get('patient_id'),
                'image_path': sample.get('image_path'),
                'mask_path':  sample.get('mask_path'),
                'has_tumour': sample.get('has_tumour', False)}


class TrainingPreprocessor(Preprocessor):
    def __init__(self, spatial_size=(96,96,96), flip_prob=0.70, rotation_prob=0.60,
                 zoom_prob=0.50, zoom_range=(0.85,1.15), gamma_prob=0.40, gamma_range=(0.70,1.40),
                 blur_prob=0.30, blur_sigma_range=(0.0,1.0), intensity_prob=0.50,
                 intensity_scale_range=(0.85,1.15), intensity_shift_range=(-0.15,0.15),
                 noise_prob=0.40, noise_std=0.10):
        super().__init__(spatial_size=spatial_size)
        self.flip_prob=flip_prob; self.rotation_prob=rotation_prob
        self.zoom_prob=zoom_prob; self.zoom_range=zoom_range
        self.gamma_prob=gamma_prob; self.gamma_range=gamma_range
        self.blur_prob=blur_prob; self.blur_sigma_range=blur_sigma_range
        self.intensity_prob=intensity_prob
        self.intensity_scale_range=intensity_scale_range
        self.intensity_shift_range=intensity_shift_range
        self.noise_prob=noise_prob; self.noise_std=noise_std

    @staticmethod
    def _gaussian_blur_3d(volume, sigma):
        if sigma < 1e-3: return volume
        radius = max(1, int(round(3 * sigma)))
        ks = 2 * radius + 1
        coords = torch.arange(ks, dtype=torch.float32) - radius
        kernel = torch.exp(-0.5 * (coords / sigma) ** 2)
        kernel = kernel / kernel.sum()
        C = volume.shape[0]
        result = volume.clone()
        vol5d = result.unsqueeze(0)
        for w_shape, pad_shape in [
            ((C,1,ks,1,1),(radius,0,0)),
            ((C,1,1,ks,1),(0,radius,0)),
            ((C,1,1,1,ks),(0,0,radius)),
        ]:
            k3d = kernel.to(volume.device).expand(C,-1).reshape(*w_shape)
            vol5d = F.conv3d(vol5d, k3d, padding=pad_shape, groups=C)
        return vol5d.squeeze(0)

    def _apply_augmentations(self, image, mask):
        for dim in (1,2,3):
            if torch.rand(1).item() < self.flip_prob:
                image = image.flip(dim); mask = mask.flip(dim)
        if torch.rand(1).item() < self.rotation_prob:
            k = int(torch.randint(1,4,(1,)).item())
            dims = [(1,2),(1,3),(2,3)][int(torch.randint(0,3,(1,)).item())]
            image = torch.rot90(image,k=k,dims=dims); mask = torch.rot90(mask,k=k,dims=dims)
        if torch.rand(1).item() < self.zoom_prob:
            lo,hi = self.zoom_range; scale = lo + torch.rand(1).item()*(hi-lo)
            d,h,w = image.shape[1],image.shape[2],image.shape[3]
            nd,nh,nw = max(1,int(round(d*scale))),max(1,int(round(h*scale))),max(1,int(round(w*scale)))
            image = F.interpolate(F.interpolate(image.unsqueeze(0),(nd,nh,nw),mode='trilinear',align_corners=False),(d,h,w),mode='trilinear',align_corners=False).squeeze(0)
            mask  = (F.interpolate(F.interpolate(mask.unsqueeze(0),(nd,nh,nw),mode='nearest'),(d,h,w),mode='nearest').squeeze(0) > 0).float()
        if torch.rand(1).item() < self.gamma_prob:
            lo,hi = self.gamma_range; gamma = lo + torch.rand(1).item()*(hi-lo)
            mn,mx = image.min(),image.max()
            if (mx-mn) > 1e-6:
                image = torch.clamp((image-mn)/(mx-mn),0,1)**gamma*(mx-mn)+mn
        if torch.rand(1).item() < self.blur_prob:
            lo,hi = self.blur_sigma_range; sigma = lo+torch.rand(1).item()*(hi-lo)
            image = self._gaussian_blur_3d(image, sigma)
        if torch.rand(1).item() < self.intensity_prob:
            lo_s,hi_s = self.intensity_scale_range; lo_b,hi_b = self.intensity_shift_range
            scale = lo_s+torch.rand(1).item()*(hi_s-lo_s); shift = lo_b+torch.rand(1).item()*(hi_b-lo_b)
            image = image*scale+shift
        if torch.rand(1).item() < self.noise_prob:
            image = image + torch.randn_like(image)*self.noise_std
        return image, mask

    def __call__(self, sample):
        processed = super().__call__(sample)
        image, mask = self._apply_augmentations(processed['image'], processed['mask'])
        processed['image'] = image; processed['mask'] = mask
        return processed


In [ ]:
%%writefile src/data_loader.py
from __future__ import annotations
import logging, random
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple
import nibabel as nib
import numpy as np
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from .preprocess import Preprocessor

LOGGER = logging.getLogger(__name__)

class BraTSDataset(Dataset):
    def __init__(self, data_dir, patients=None, transform=None):
        self.data_dir = Path(data_dir); self.transform = transform or Preprocessor()
        self.patients = patients or [p.name for p in sorted(self.data_dir.iterdir()) if p.is_dir()]
        self.samples = []
        for pid in self.patients:
            pd = self.data_dir / pid
            t1c = next(pd.glob('*_t1c.nii.gz'), None)
            msk = next(pd.glob('*_gtv.nii.gz'), None)
            if t1c and msk:
                self.samples.append({'patient_id':pid,'image_path':str(t1c),'mask_path':str(msk),'has_tumour':True})
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        image = nib.load(s['image_path']).get_fdata(dtype=np.float32)
        mask  = nib.load(s['mask_path']).get_fdata(dtype=np.float32)
        payload = {'image':image,'mask':mask,'patient_id':s['patient_id'],
                   'image_path':s['image_path'],'mask_path':s['mask_path'],'has_tumour':s['has_tumour']}
        return self.transform(payload) if self.transform else payload

class IXIDataset(Dataset):
    def __init__(self, data_dir, patients=None, transform=None):
        self.data_dir = Path(data_dir); self.transform = transform or Preprocessor()
        all_dirs = [p.name for p in sorted(self.data_dir.iterdir()) if p.is_dir()]
        self.patients = patients if patients is not None else all_dirs
        self.samples = []
        for pid in self.patients:
            pd = self.data_dir / pid
            nii = next(pd.glob('*.nii'), None) or next(pd.glob('*.nii.gz'), None)
            if nii:
                self.samples.append({'patient_id':pid,'image_path':str(nii),'mask_path':None,'has_tumour':False})
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        image = nib.load(s['image_path']).get_fdata(dtype=np.float32)
        mask  = np.zeros(image.shape, dtype=np.float32)
        payload = {'image':image,'mask':mask,'patient_id':s['patient_id'],
                   'image_path':s['image_path'],'mask_path':None,'has_tumour':False}
        return self.transform(payload) if self.transform else payload

def _split_ids(ids, val_frac, test_frac, rng):
    ids = list(ids); rng.shuffle(ids); n = len(ids)
    n_val  = max(1, int(n*val_frac))  if n >= 3 else 0
    n_test = max(1, int(n*test_frac)) if n >= 3 else 0
    while n_val+n_test >= n and (n_val>0 or n_test>0):
        if n_val >= n_test and n_val > 0: n_val -= 1
        elif n_test > 0: n_test -= 1
    return ids[:n-n_val-n_test], ids[n-n_val-n_test:n-n_test], ids[n-n_test:]

class CombinedDataModule:
    def __init__(self, brats_dir, ixi_dir, batch_size=1, validation_split=0.10,
                 test_split=0.10, train_transform=None, eval_transform=None,
                 max_brats_patients=None, max_ixi_patients=None, random_seed=42):
        self.brats_dir=brats_dir; self.ixi_dir=ixi_dir; self.batch_size=batch_size
        self.train_transform=train_transform or Preprocessor()
        self.eval_transform=eval_transform or Preprocessor()
        rng = random.Random(random_seed)
        brats_ids = [p.name for p in sorted(Path(brats_dir).iterdir()) if p.is_dir()]
        rng.shuffle(brats_ids)
        if max_brats_patients: brats_ids = brats_ids[:max_brats_patients]
        self.brats_train,self.brats_val,self.brats_test = _split_ids(brats_ids,validation_split,test_split,random.Random(random_seed+1))
        ixi_ids = [p.name for p in sorted(Path(ixi_dir).iterdir()) if p.is_dir()]
        rng.shuffle(ixi_ids)
        if max_ixi_patients: ixi_ids = ixi_ids[:max_ixi_patients]
        self.ixi_train,self.ixi_val,self.ixi_test = _split_ids(ixi_ids,validation_split,test_split,random.Random(random_seed+2))
        LOGGER.info('Split — BraTS train=%d val=%d test=%d | IXI train=%d val=%d test=%d',
            len(self.brats_train),len(self.brats_val),len(self.brats_test),
            len(self.ixi_train),len(self.ixi_val),len(self.ixi_test))
    def train_dataloader(self):
        ds = ConcatDataset([BraTSDataset(self.brats_dir,self.brats_train,self.train_transform),
                            IXIDataset(self.ixi_dir,self.ixi_train,self.train_transform)])
        LOGGER.info('Train: %d total samples', len(ds))
        return DataLoader(ds, batch_size=self.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    def val_dataloader(self):
        ds = ConcatDataset([BraTSDataset(self.brats_dir,self.brats_val,self.eval_transform),
                            IXIDataset(self.ixi_dir,self.ixi_val,self.eval_transform)])
        return DataLoader(ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True) if len(ds)>0 else None
    def test_dataloader(self):
        ds = ConcatDataset([BraTSDataset(self.brats_dir,self.brats_test,self.eval_transform),
                            IXIDataset(self.ixi_dir,self.ixi_test,self.eval_transform)])
        return DataLoader(ds, batch_size=1, shuffle=False, num_workers=2) if len(ds)>0 else None


In [ ]:
%%writefile src/trainer.py
from __future__ import annotations
import logging, math, os
from pathlib import Path
from typing import Any, Dict, Optional
import matplotlib.pyplot as plt
import torch
from monai.losses import DiceFocalLoss, TverskyLoss
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
try:
    from torch.utils.tensorboard import SummaryWriter
    _HAS_TB = True
except ImportError:
    _HAS_TB = False
from .model import get_model
from .utils import (compute_dice,compute_f1,compute_iou,compute_pixel_accuracy,
                    compute_precision,compute_recall,ensure_directory,get_device,save_json)

LOGGER = logging.getLogger(__name__)

class SegmentationTrainer:
    def __init__(self, config):
        self.config = config
        self.device = get_device()
        self.accumulation_steps = max(1, int(config.get('gradient_accumulation_steps', 4)))
        self.model = get_model(
            architecture=config.get('architecture','segresnet'),
            in_channels=config.get('in_channels',1),
            out_channels=config.get('out_channels',2),
        ).to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=config.get('learning_rate',2e-4),
                               weight_decay=1e-5, betas=(0.9,0.999), eps=1e-8)
        self.dice_focal_loss_fn = DiceFocalLoss(
            include_background=False, to_onehot_y=True, softmax=True,
            gamma=2.0, lambda_dice=0.5, lambda_focal=0.5)
        self.tversky_loss_fn = TverskyLoss(
            include_background=False, to_onehot_y=True, softmax=True,
            alpha=0.3, beta=0.7)
        total_epochs = config.get('epochs',100)
        warmup_epochs = config.get('warmup_epochs',5)
        self._warmup_epochs = warmup_epochs; self._warmup_done = False
        self.warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer, lr_lambda=lambda e: float(e+1)/float(max(1,warmup_epochs)) if e < warmup_epochs else 1.0)
        self.cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=config.get('lr_restart_period',25), T_mult=2, eta_min=1e-7)
        self.scaler = GradScaler(device=self.device.type)
        self.writer = SummaryWriter(log_dir=config.get('log_dir','output/runs')) if _HAS_TB else None
        self.best_dice = 0.0; self.epochs_without_improvement = 0; self._seen_nonzero_dice = False
        self.checkpoint_path = Path(config.get('checkpoint_path','models/best_model.pth')).expanduser().resolve()
        self.checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        self.history = {k:[] for k in ('train_loss','val_loss','train_accuracy','val_accuracy',
                                        'val_dice','val_iou','val_precision','val_recall',
                                        'val_f1','val_cls_accuracy','lr')}

    def _loss(self, outputs, target):
        return 0.5*self.dice_focal_loss_fn(outputs,target) + 0.5*self.tversky_loss_fn(outputs,target)

    def train(self, train_loader, val_loader):
        if cp := self.config.get('resume_from'):
            self._load_checkpoint(cp)
        for epoch in range(self.config.get('epochs',100)):
            tl, ta = self._train_epoch(train_loader, epoch)
            self.history['train_loss'].append(tl); self.history['train_accuracy'].append(ta)
            lr = self.optimizer.param_groups[0]['lr']; self.history['lr'].append(lr)
            if self.writer: self.writer.add_scalar('train/loss',tl,epoch); self.writer.add_scalar('train/lr',lr,epoch)
            vs = self._validate(val_loader,epoch) if (val_loader and len(val_loader)>0) else self._empty()
            for k in ('val_loss','val_accuracy','val_dice','val_iou','val_precision','val_recall','val_f1','val_cls_accuracy'):
                self.history[k].append(vs[k])
            if epoch < self._warmup_epochs: self.warmup_scheduler.step()
            else: self.cosine_scheduler.step(epoch-self._warmup_epochs)
            self._save_checkpoint(epoch, vs['val_dice'])
            if self._should_stop(vs['val_dice']): LOGGER.info('Early stopping at epoch %d',epoch+1); break
        self._save_plots(); return self.history

    def _train_epoch(self, loader, epoch):
        self.model.train(); total_loss=0.; total_acc=0.; n=len(loader)
        self.optimizer.zero_grad(set_to_none=True)
        for i, batch in enumerate(loader):
            imgs = batch['image'].to(self.device); msks = batch['mask'].to(self.device)
            if imgs.ndim==4: imgs=imgs.unsqueeze(1)
            if msks.ndim==4: msks=msks.unsqueeze(1)
            with autocast(device_type=self.device.type, enabled=self.device.type=='cuda'):
                out = self.model(imgs); tgt = msks[:,0].long().unsqueeze(1)
                loss = self._loss(out, tgt) / self.accumulation_steps
            self.scaler.scale(loss).backward()
            total_loss += loss.item()*self.accumulation_steps
            with torch.no_grad():
                total_acc += compute_pixel_accuracy(torch.argmax(out,dim=1), tgt.squeeze(1))
            if (i+1)%self.accumulation_steps==0 or i==n-1:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer); self.scaler.update()
                self.optimizer.zero_grad(set_to_none=True)
        avg_l=total_loss/max(1,n); avg_a=total_acc/max(1,n)
        LOGGER.info('Epoch %3d | loss=%.4f | acc=%.4f | lr=%.2e',epoch+1,avg_l,avg_a,self.optimizer.param_groups[0]['lr'])
        return avg_l, avg_a

    @staticmethod
    def _cls_acc(preds, has_tumour_batch):
        B=preds.shape[0]; correct=0
        for i in range(B):
            pp=(preds[i]>0).any().item()
            gp=bool(has_tumour_batch[i].item()) if torch.is_tensor(has_tumour_batch[i]) else bool(has_tumour_batch[i])
            if pp==gp: correct+=1
        return correct/B if B>0 else 0.0

    def _validate(self, loader, epoch):
        self.model.eval(); tot={k:0. for k in ('loss','accuracy','dice','iou','precision','recall','f1','cls_accuracy')}; n=0
        with torch.no_grad():
            for batch in loader:
                imgs=batch['image'].to(self.device); msks=batch['mask'].to(self.device)
                ht=batch.get('has_tumour',None)
                if imgs.ndim==4: imgs=imgs.unsqueeze(1)
                if msks.ndim==4: msks=msks.unsqueeze(1)
                out=self.model(imgs); tgt=msks[:,0].long().unsqueeze(1)
                preds=torch.argmax(out,dim=1); tl=tgt.squeeze(1)
                pb=(preds>0).float(); tb=(tl>0).float()
                tot['loss']+=self._loss(out,tgt).item()
                tot['accuracy']+=compute_pixel_accuracy(preds,tl)
                tot['dice']+=compute_dice(pb,tb); tot['iou']+=compute_iou(pb,tb)
                tot['precision']+=compute_precision(pb,tb); tot['recall']+=compute_recall(pb,tb)
                tot['f1']+=compute_f1(pb,tb)
                if ht is not None: tot['cls_accuracy']+=self._cls_acc(preds,ht)
                n+=1
        if n==0: return self._empty()
        vs={f'val_{k}':v/n for k,v in tot.items()}
        if self.writer:
            for k,v in vs.items(): self.writer.add_scalar(k.replace('val_','val/'),v,epoch)
        LOGGER.info('Epoch %3d | val_loss=%.4f | dice=%.4f | iou=%.4f | prec=%.4f | rec=%.4f | f1=%.4f | cls_acc=%.4f',
            epoch+1,vs['val_loss'],vs['val_dice'],vs['val_iou'],vs['val_precision'],vs['val_recall'],vs['val_f1'],vs['val_cls_accuracy'])
        return vs

    @staticmethod
    def _empty():
        return {f'val_{k}':0. for k in ('loss','accuracy','dice','iou','precision','recall','f1','cls_accuracy')}

    def _load_checkpoint(self, path):
        if path and os.path.exists(path):
            s=torch.load(path,map_location=self.device)
            self.model.load_state_dict(s['model_state_dict'])
            self.optimizer.load_state_dict(s['optimizer_state_dict'])
            self.best_dice=s.get('best_dice',0.0)
            LOGGER.info('Resumed from %s',path)

    def _save_checkpoint(self, epoch, val_dice):
        improved = val_dice > self.best_dice
        if improved: self.best_dice=val_dice; self.epochs_without_improvement=0; self._seen_nonzero_dice=True
        elif self._seen_nonzero_dice: self.epochs_without_improvement+=1
        else: self.epochs_without_improvement=0
        ckpt={'model_state_dict':self.model.state_dict(),'optimizer_state_dict':self.optimizer.state_dict(),
              'best_dice':self.best_dice,'architecture':self.config.get('architecture','segresnet'),
              'epoch':epoch,'history':self.history}
        if improved:
            torch.save(ckpt, self.checkpoint_path)
            LOGGER.info('✓ New best Dice=%.4f — saved to %s', val_dice, self.checkpoint_path)
        torch.save(ckpt, self.checkpoint_path.with_suffix('.bak'))

    def _should_stop(self, val_dice):
        return self.epochs_without_improvement >= self.config.get('early_stopping_patience',15)

    def _save_plots(self):
        pd = ensure_directory('results/plots')
        ep = list(range(1, len(self.history['train_loss'])+1))
        def pl(keys, title, ylabel, fname):
            fig,ax=plt.subplots(figsize=(10,5))
            for k,lbl in keys:
                d=self.history.get(k,[])
                if d: ax.plot(ep[:len(d)],d,marker='o',ms=3,label=lbl)
            ax.set_title(title);ax.set_xlabel('Epoch');ax.set_ylabel(ylabel);ax.legend();fig.tight_layout()
            fig.savefig(pd/fname,dpi=150);plt.close(fig)
        pl([('train_loss','Train'),('val_loss','Val')],'Loss','Loss','loss.png')
        pl([('val_dice','Dice'),('val_iou','IoU'),('val_f1','F1'),('val_cls_accuracy','Cls Acc')],'Metrics','Score','metrics.png')
        pl([('val_precision','Precision'),('val_recall','Recall')],'Precision & Recall','Score','prec_rec.png')

    def save_history(self, path): save_json(path, self.history)


In [ ]:
%%writefile train.py
from __future__ import annotations
import argparse, logging, shutil
from pathlib import Path
from src.data_loader import CombinedDataModule
from src.preprocess import Preprocessor, TrainingPreprocessor
from src.trainer import SegmentationTrainer
from src.utils import configure_logging, ensure_directory

PROJECT_ROOT = Path(__file__).resolve().parent

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--architecture', default='segresnet', choices=['segresnet','unet','dynunet'])
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--batch-size', type=int, default=1)
    p.add_argument('--gradient-accumulation-steps', type=int, default=4)
    p.add_argument('--data-dir', required=True)
    p.add_argument('--healthy-dir', required=True)
    p.add_argument('--spatial-size', type=int, nargs=3, default=(96,96,96))
    p.add_argument('--validation-split', type=float, default=0.10)
    p.add_argument('--test-split', type=float, default=0.10)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--learning-rate', type=float, default=2e-4)
    p.add_argument('--warmup-epochs', type=int, default=5)
    p.add_argument('--lr-restart-period', type=int, default=25)
    p.add_argument('--early-stopping-patience', type=int, default=15)
    p.add_argument('--resume-from', default=None)
    p.add_argument('--max-brats-patients', type=int, default=None)
    p.add_argument('--max-ixi-patients', type=int, default=None)
    return p.parse_args()

def main():
    args = parse_args()
    configure_logging('INFO')
    logger = logging.getLogger(__name__)
    logger.info('Training | arch=%s | epochs=%d | spatial=%s', args.architecture, args.epochs, args.spatial_size)
    ensure_directory(PROJECT_ROOT/'output'); ensure_directory(PROJECT_ROOT/'models')
    ensure_directory(PROJECT_ROOT/'results'/'plots')
    spatial = tuple(args.spatial_size)
    dm = CombinedDataModule(
        brats_dir=args.data_dir, ixi_dir=args.healthy_dir,
        batch_size=args.batch_size, validation_split=args.validation_split,
        test_split=args.test_split,
        train_transform=TrainingPreprocessor(spatial_size=spatial),
        eval_transform=Preprocessor(spatial_size=spatial),
        max_brats_patients=args.max_brats_patients,
        max_ixi_patients=args.max_ixi_patients,
        random_seed=args.seed,
    )
    config = {
        'architecture': args.architecture, 'epochs': args.epochs,
        'batch_size': args.batch_size, 'learning_rate': args.learning_rate,
        'warmup_epochs': args.warmup_epochs, 'lr_restart_period': args.lr_restart_period,
        'gradient_accumulation_steps': args.gradient_accumulation_steps,
        'checkpoint_path': str(PROJECT_ROOT/'models'/'best_model.pth'),
        'log_dir': str(PROJECT_ROOT/'output'/'runs'),
        'resume_from': args.resume_from,
        'early_stopping_patience': args.early_stopping_patience,
    }
    trainer = SegmentationTrainer(config)
    trainer.train(dm.train_dataloader(), dm.val_dataloader())
    trainer.save_history(str(PROJECT_ROOT/'results'/'training_history.json'))
    logger.info('Done. Best model → %s', config['checkpoint_path'])

if __name__ == '__main__': main()


In [ ]:
# Quick sanity check — make sure all imports work
import sys
sys.path.insert(0, '/content/BrainTumour')
from src.model import get_model
from src.preprocess import TrainingPreprocessor, Preprocessor
from src.trainer import SegmentationTrainer
from src.data_loader import CombinedDataModule
print('✅ All source files imported successfully')

## ✅ Step 4 — Keep-Alive Script (Run this to prevent Colab disconnecting)

Paste this in your browser console (`F12` → Console tab) to prevent the session from timing out:

In [ ]:
# Run this cell — it prints a JS snippet. Copy it into your browser console.
print("""Paste this in browser F12 → Console:

function keepAlive() {
    var btn = document.querySelector('colab-toolbar-button[id=\"connect\"]');
    if (btn) { btn.click(); console.log('keep-alive ping'); }
}
setInterval(keepAlive, 55000);
console.log('Keep-alive started. Session will stay active.');
""")

## ✅ Step 5 — START TRAINING 🚀

In [ ]:
import subprocess, sys

# These should already be set from Step 2 — change if needed
BRATS_DIR = '/content/drive/MyDrive/Tumour-MRI-Data/BraTS-MEN-RT-Train-v2'
IXI_DIR   = '/content/drive/MyDrive/Tumour-MRI-Data/IXI_T1'

cmd = [
    sys.executable, 'train.py',
    '--data-dir',    BRATS_DIR,
    '--healthy-dir', IXI_DIR,
    '--architecture', 'segresnet',
    '--epochs',      '100',
    '--spatial-size', '96', '96', '96',
    '--learning-rate', '2e-4',
    '--warmup-epochs', '5',
    '--lr-restart-period', '25',
    '--gradient-accumulation-steps', '4',
    '--early-stopping-patience', '15',
    '--validation-split', '0.10',
    '--test-split', '0.10',
]

print('Starting training...')
print(' '.join(cmd))
print('-' * 60)

# Stream output line by line so you see progress in real time
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, cwd='/content/BrainTumour')
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nTraining finished with exit code: {proc.returncode}')

## ✅ Step 6 — Save Model & Results to Google Drive

Run this after training completes (or at any point to save a mid-run checkpoint).

In [ ]:
import shutil, os, json
from pathlib import Path
import torch

SAVE_DIR = '/content/drive/MyDrive/Tumour-MRI-Data'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save best model
best_model = '/content/BrainTumour/models/best_model.pth'
if os.path.exists(best_model):
    shutil.copy(best_model, f'{SAVE_DIR}/best_model.pth')
    ckpt = torch.load(best_model, map_location='cpu')
    print(f'✅ Model saved to Drive')
    print(f'   Best Dice: {ckpt["best_dice"]:.4f}')
    print(f'   Architecture: {ckpt["architecture"]}')
    print(f'   Epoch: {ckpt["epoch"]}')
else:
    # Try backup
    backup = '/content/BrainTumour/models/best_model.bak'
    if os.path.exists(backup):
        shutil.copy(backup, f'{SAVE_DIR}/best_model.pth')
        print('✅ Backup checkpoint saved to Drive')
    else:
        print('❌ No checkpoint found yet — training may not have started')

# Save training history
history_path = '/content/BrainTumour/results/training_history.json'
if os.path.exists(history_path):
    shutil.copy(history_path, f'{SAVE_DIR}/training_history.json')
    with open(history_path) as f:
        h = json.load(f)
    print(f'\n📊 Final Metrics (last epoch):')
    for k in ('val_dice','val_iou','val_precision','val_recall','val_f1','val_cls_accuracy','val_accuracy'):
        vals = h.get(k,[])
        if vals: print(f'   {k}: {vals[-1]:.4f}  (best: {max(vals):.4f})')

# Save plots
plots_dir = '/content/BrainTumour/results/plots'
if os.path.isdir(plots_dir):
    os.makedirs(f'{SAVE_DIR}/plots', exist_ok=True)
    for f in os.listdir(plots_dir):
        shutil.copy(f'{plots_dir}/{f}', f'{SAVE_DIR}/plots/{f}')
    print(f'\n✅ Training plots saved to Drive/Tumour-MRI-Data/plots/')

## ✅ Step 7 — Resume if Colab Disconnected

If Colab times out mid-training:
1. Re-run **Steps 1, 2, 3** (GPU check, mount Drive, write code)
2. Run the cell below to copy your saved checkpoint back
3. Re-run **Step 5** — it will resume from where it left off

In [ ]:
import shutil, os, torch

SAVE_DIR = '/content/drive/MyDrive/Tumour-MRI-Data'
os.makedirs('/content/BrainTumour/models', exist_ok=True)

saved_model = f'{SAVE_DIR}/best_model.pth'
if os.path.exists(saved_model):
    shutil.copy(saved_model, '/content/BrainTumour/models/best_model.pth')
    ckpt = torch.load(saved_model, map_location='cpu')
    print(f'✅ Checkpoint restored from Drive')
    print(f'   Resuming from epoch {ckpt["epoch"]+1}')
    print(f'   Best Dice so far: {ckpt["best_dice"]:.4f}')
else:
    print('No saved model found on Drive yet.')

print('\nNow go back and run Step 5 — training will resume automatically.')

## ✅ Step 8 — Download Trained Model to Your PC

After training, download the model directly to your computer:

In [ ]:
from google.colab import files
import os

model_path = '/content/BrainTumour/models/best_model.pth'
drive_path = '/content/drive/MyDrive/Tumour-MRI-Data/best_model.pth'

# Try local first, then Drive
download_path = model_path if os.path.exists(model_path) else drive_path

if os.path.exists(download_path):
    print(f'Downloading from {download_path}...')
    files.download(download_path)
    print('✅ Download started — check your browser downloads')
else:
    print('❌ No model file found. Run training first.')

## ✅ Step 9 — Quick Test Inference (After Training)

After downloading the model back to your PC, use inference.py locally:

```cmd
python inference.py --input path\to\any\brain.nii.gz
```

Or test directly in Colab on a validation sample:

In [ ]:
import torch, sys, os
import nibabel as nib
import numpy as np

sys.path.insert(0, '/content/BrainTumour')
from src.model import get_model
from src.preprocess import Preprocessor
from src.utils import get_device

# Load model
device = get_device()
ckpt = torch.load('/content/BrainTumour/models/best_model.pth', map_location=device)
model = get_model(architecture=ckpt['architecture']).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Model loaded | Architecture: {ckpt["architecture"]} | Best Dice: {ckpt["best_dice"]:.4f}')

# Pick the first BraTS validation sample to test on
BRATS_DIR = '/content/drive/MyDrive/Tumour-MRI-Data/BraTS-MEN-RT-Train-v2'
sample_dir = sorted(os.listdir(BRATS_DIR))[0]
sample_path = next((BRATS_DIR+'/'+sample_dir+'/'+f for f in os.listdir(BRATS_DIR+'/'+sample_dir) if '_t1c' in f), None)
print(f'Testing on: {sample_dir}')

# Run inference
image = nib.load(sample_path).get_fdata(dtype=np.float32)
prep = Preprocessor(spatial_size=(96,96,96))
sample = prep({'image': image, 'mask': np.zeros_like(image), 'patient_id': sample_dir})
img_tensor = sample['image'].unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(img_tensor)
    probs = torch.softmax(logits, dim=1)[:, 1].squeeze()

max_prob = probs.max().item()
tumour_voxels = (probs > 0.5).sum().item()
verdict = 'TUMOUR DETECTED' if tumour_voxels > 10 else 'NO TUMOUR'

print(f'\n{'='*40}')
print(f'  VERDICT: {verdict}')
print(f'  Max tumour probability: {max_prob:.4f}')
print(f'  Tumour voxels (>0.5): {tumour_voxels:,}')
print(f'  Tumour volume: {tumour_voxels * 0.001:.3f} cm³')
print(f'{'='*40}')